# CN7 가스 F1 0.7 개발 후보 검증
모든 게이트/혼동행렬 재계산, 모든 기존 fit audit 확인, 새175 분류기의 3개 평가구간 라벨 반전·새 학습 출력 불변성 점검.
새 Run 독립검증을 의미하지 않는다. 최종 후보는177 confirmed_RF_LR, 냉각 정상통계 모델은 제외.


In [1]:
import nbformat
from pathlib import Path
import json,hashlib
import numpy as np,pandas as pd
from sklearn.metrics import confusion_matrix,f1_score
ROOT=Path.cwd();OUT=ROOT/"reports/tables"
SOURCE=OUT/"cn7_reference_snapshot"
if not SOURCE.exists(): SOURCE=Path(r"C:/Users/hayeon/.codex/worktrees/b365/KAMP-IMM/reports/tables")
d=pd.read_csv(SOURCE/"cn7_163_gas_supervised_input_raw_shots.csv",parse_dates=["TimeStamp"]).sort_values(["run_id","TimeStamp"]).reset_index(drop=True)
a=pd.read_csv(OUT/"cn7_177_confirmed_expert_predictions.csv",parse_dates=["TimeStamp"]).set_index("TimeStamp").loc[d.TimeStamp]
assert len(d)==606 and d.TimeStamp.is_unique
assert d.gas.sum()==13 and d.any_fail.eq(0).sum()==592
y=d.gas.to_numpy(int);eligible=(d.gas|d.any_fail.eq(0)).to_numpy()
expected=a.base173.to_numpy(int)|(a.raw_RF.to_numpy(bool)&a.speed_high.to_numpy(bool)).astype(int)|(a.raw_LR.to_numpy(bool)&a.meter_high.to_numpy(bool)).astype(int)
assert np.array_equal(expected,a.confirmed_RF_LR.to_numpy(int))
audit=pd.read_csv(OUT/"cn7_177_confirmation_audit.csv")
checks=[]
for held in d.time_block.unique():
 q=d.index[d.time_block.eq(held)].to_numpy()
 run=d.run_id.iloc[q[0]]
 ctx=~(d.run_id.eq(run)&d.seq.between(d.seq.iloc[q].min()-10,d.seq.iloc[q].max()+10))
 ni=d.index[ctx&d.any_fail.eq(0)&d.run_id.eq(run)].to_numpy()
 if len(ni)<8:ni=d.index[ctx&d.any_fail.eq(0)].to_numpy()
 assert set(ni).isdisjoint(q)
 for field,flag,rankcol in [("Max_Injection_Speed_raw","speed_high","speed_midrank"),("Plasticizing_Position_raw","meter_high","meter_midrank")]:
  vals=np.sort(np.round(d[field].iloc[ni].to_numpy(),4))
  query=np.round(d[field].iloc[q].to_numpy(),4)
  r=(np.searchsorted(vals,query,side="left")+np.searchsorted(vals,query,side="right"))/(2*len(vals))
  assert np.allclose(r,a[rankcol].to_numpy()[q],rtol=0,atol=1e-14)
  assert np.array_equal(r>.95,a[flag].to_numpy()[q].astype(bool))
  checks.append({"heldout_block":held,"field":field,"reference_count":len(ni),"reference_overlap":0})
met=[]
for cohort,mask in [("normal_and_gas",eligible),("all_official",np.ones(len(d),bool))]:
 tn,fp,fn,tp=confusion_matrix(y[mask],expected[mask],labels=[0,1]).ravel()
 met.append({"cohort":cohort,"N":int(mask.sum()),"TP":int(tp),"FP":int(fp),"FN":int(fn),"TN":int(tn),"F1":float(f1_score(y[mask],expected[mask]))})
assert met[0]["TP"]==10 and met[0]["FP"]==5 and met[0]["FN"]==3 and met[0]["TN"]==587
assert met[0]["F1"]>=.7
assert met[1]["TP"]==10 and met[1]["FP"]==6 and met[1]["FN"]==3 and met[1]["TN"]==587
# Verify prior fit audits, including source classifiers 163/164 and donor-supported173.
for folder,file,columns in [
 (SOURCE,"cn7_163_gas_supervised_audit.csv",["timestamp_overlap","test_raw_records_in_training_context"]),
 (SOURCE,"cn7_164_gas_supervised_audit.csv",["timestamp_overlap","test_raw_in_training_history"]),
 (OUT,"cn7_170_persistence_training_audit.csv",["timestamp_overlap","test_raw_in_training_history"]),
 (OUT,"cn7_173_transfer_audit.csv",["timestamp_overlap","test_raw_in_training_history"]),
 (OUT,"cn7_175_magnitude_training_audit.csv",["timestamp_overlap","test_raw_in_training_history"])]:
 z=pd.read_csv(folder/file)
 for c in columns:assert z[c].eq(0).all(),(file,c)
 print("PASS recorded fits",file,len(z))
# Fresh fits of new175 heads: invert labels in each query block, clear caches, refit.
# Query block AND adjacent10 records remain excluded from every reference.
n=nbformat.read(ROOT/"notebooks/175_cn7_gas_direction_magnitude_local_experts.ipynb",as_version=4)
s="\n".join(c.source for c in n.cells if c.cell_type=="code")
ns={}
exec(s[:s.index("runner=oldcode[")],ns)
probe_results=[]
for held in ["cn7_R1__B02","cn7_R1__B03","cn7_R2__B09"]:
 q=np.flatnonzero(ns["blocks"]==held)
 parent=np.ones(ns["N"],bool)
 original=ns["y"].copy()
 before,ref,ctx=ns["predict"](parent,q)
 assert not ctx[q].any()
 ns["y"][q]=1-ns["y"][q]
 for cache in ["memo","scorememo","localmemo"]:ns[cache].clear()
 after,ref2,ctx2=ns["predict"](parent,q)
 assert np.array_equal(ref,ref2) and np.array_equal(ctx,ctx2)
 maxdiff=max(float(np.max(np.abs(before[k]-after[k]))) for k in before)
 assert maxdiff<=1e-12 # Parallel tree reductions can differ at floating-point roundoff.
 ns["y"][:]=original
 for cache in ["memo","scorememo","localmemo"]:ns[cache].clear()
 probe_results.append({"heldout_block":held,"heads":len(before),"max_prediction_difference":maxdiff,"test_training_overlap":0})
 print("PASS fresh-fit query-label invariance",held,len(before))
# Diagnostics never redefine the evaluation cohort.
case=d[["TimeStamp","gas","any_fail","run_id","seq"]].copy()
case["gas_prediction"]=expected
case["base173"]=a.base173.to_numpy(int)
case["localRF_fast"]=a.localRF_fast.to_numpy(int)
case["absoluteLR_meter"]=a.absoluteLR_meter.to_numpy(int)
case["speed_midrank"]=a.speed_midrank.to_numpy(float)
case["meter_midrank"]=a.meter_midrank.to_numpy(float)
case.to_csv(OUT/"cn7_178_verified_best_predictions.csv",index=False,encoding="utf-8-sig")
case[case.gas].to_csv(OUT/"cn7_178_verified_all13_gas.csv",index=False,encoding="utf-8-sig")
case[case.any_fail.eq(0)&case.gas_prediction.eq(1)].to_csv(OUT/"cn7_178_verified_false_positives.csv",index=False,encoding="utf-8-sig")
pd.DataFrame(met).to_csv(OUT/"cn7_178_verified_metrics.csv",index=False)
proof={"verified_metrics":met,"confirmation_recomputed":len(checks),"query_label_fresh_fit_probes":probe_results,
 "input_sha256":hashlib.sha256((SOURCE/"cn7_163_gas_supervised_input_raw_shots.csv").read_bytes()).hexdigest(),
 "predictions_sha256":hashlib.sha256((OUT/"cn7_178_verified_best_predictions.csv").read_bytes()).hexdigest(),
 "caveat":"partial fresh-fit probes plus all saved audits; not a proof of untouched independent validation. Post-hoc feature/gate/combination design persists.",
 "future_validation":"same-run temporal folds use future training records; not forward causal validation"}
(OUT/"cn7_178_completion_check.json").write_text(json.dumps(proof,ensure_ascii=False,indent=2),encoding="utf-8")
print("\nVERIFIED METRICS",met)
print("\nALL GAS\n",case[case.gas].to_string(index=False))
print("\nFALSE POSITIVES\n",case[case.any_fail.eq(0)&case.gas_prediction.eq(1)].to_string(index=False))
print("\nMISSED\n",case[case.gas&case.gas_prediction.eq(0)].to_string(index=False))


PASS recorded fits cn7_163_gas_supervised_audit.csv 220
PASS recorded fits cn7_164_gas_supervised_audit.csv 220
PASS recorded fits cn7_170_persistence_training_audit.csv 220
PASS recorded fits cn7_173_transfer_audit.csv 220
PASS recorded fits cn7_175_magnitude_training_audit.csv 220


PASS fresh-fit query-label invariance cn7_R1__B02 16


PASS fresh-fit query-label invariance cn7_R1__B03 16


PASS fresh-fit query-label invariance cn7_R2__B09 16

VERIFIED METRICS [{'cohort': 'normal_and_gas', 'N': 605, 'TP': 10, 'FP': 5, 'FN': 3, 'TN': 587, 'F1': 0.7142857142857143}, {'cohort': 'all_official', 'N': 606, 'TP': 10, 'FP': 6, 'FN': 3, 'TN': 587, 'F1': 0.6896551724137931}]

ALL GAS
           TimeStamp  gas  any_fail run_id  seq  gas_prediction  base173  localRF_fast  absoluteLR_meter  speed_midrank  meter_midrank
2020-10-16 05:21:38 True         1 cn7_R1   24               1        1             0                 0       0.208029       0.448905
2020-10-16 05:37:29 True         1 cn7_R1   40               0        0             0                 0       0.077703       0.442568
2020-10-16 05:45:26 True         1 cn7_R1   48               1        1             0                 0       0.077703       0.442568
2020-10-16 05:47:24 True         1 cn7_R1   50               1        1             0                 0       0.418919       0.442568
2020-10-16 05:48:24 True         1 cn7_R